In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch10-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "group_cv.py"

## 骨架分組的五折驗證

比較同一份 ESOL 開發資料上的四個樹深。測試集保留到選定樹深後只評估一次。程式格採用 Colab 獨立環境執行，首次需下載公開 ESOL 資料。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import deepchem as dc
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "book-moleculenet"))
cache.mkdir(parents=True, exist_ok=True)
tasks, (train, valid, test), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
    transformers=[], data_dir=str(cache), save_dir=str(cache), reload=False,
)

def structure_columns(ids):
    mols = [Chem.MolFromSmiles(s) for s in ids]
    X = np.array([[Descriptors.MolWt(m), Descriptors.MolLogP(m),
                   Descriptors.TPSA(m), Descriptors.NumRotatableBonds(m)] for m in mols])
    groups = np.array([MurckoScaffold.MurckoScaffoldSmiles(mol=m) for m in mols])
    return X, groups

X_dev, groups = structure_columns(np.concatenate([train.ids, valid.ids]))
y_dev = np.concatenate([train.y[:, 0], valid.y[:, 0]])
X_test, test_groups = structure_columns(test.ids)
assert set(groups).isdisjoint(test_groups)
folds = list(GroupKFold(n_splits=5).split(X_dev, y_dev, groups))
depths = [1, 3, 6, None]
scores = {}
for depth in depths:
    fold_scores = []
    for fit, hold in folds:
        assert set(groups[fit]).isdisjoint(groups[hold])
        tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
        tree.fit(X_dev[fit], y_dev[fit])
        fold_scores.append((mean_absolute_error(y_dev[fit], tree.predict(X_dev[fit])),
                            mean_absolute_error(y_dev[hold], tree.predict(X_dev[hold]))))
    scores[depth] = np.array(fold_scores)
    print("depth", depth, "mean train/CV MAE", scores[depth].mean(axis=0).round(4))
best = min(depths, key=lambda depth: scores[depth][:, 1].mean())
final = DecisionTreeRegressor(max_depth=best, random_state=42).fit(X_dev, y_dev)
print("development/test", len(y_dev), len(test))
print("validation fold sizes", [len(hold) for _, hold in folds])
print("selected depth", best, "fold MAEs", scores[best][:, 1].round(4))
print("final test MAE", round(mean_absolute_error(test.y[:, 0], final.predict(X_test)), 4))
fig, ax = plt.subplots(figsize=(7, 4.3), layout="constrained")
positions = np.arange(len(depths))
ax.plot(positions, [scores[d][:, 0].mean() for d in depths], "o-", label="Training (mean)")
ax.plot(positions, [scores[d][:, 1].mean() for d in depths], "s-", label="Validation (mean)")
for fold in range(len(folds)):
    ax.scatter(positions + (fold - 2) * .035, [scores[d][fold, 1] for d in depths],
               color="0.42", s=16, alpha=.64, label="Individual folds" if fold == 0 else None)
ax.set(xticks=positions, xticklabels=["1", "3", "6", "Unlimited"],
       xlabel="Maximum tree depth", ylabel="MAE in log solubility")
ax.legend(frameon=False)
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
fig.savefig(out / "10-03-cv-depth.png", dpi=160)

In [ ]:
from IPython.display import Image, display
subprocess.run([python, str(script)], check=True)
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
display(Image(filename=str(out / "10-03-cv-depth.png")))